# Final test — execute once after development

This notebook is intentionally separate. It loads a frozen decision from the development notebook, verifies the dataset checksum, refits both chosen models on all published train and validation rows, then evaluates all published test rows. It does not select a model or threshold using test outcomes. The initial guard is disabled by default.

**Historical limitation:** this publication's test was inspected in earlier project work. This new workflow prevents further test-guided changes, but cannot undo that earlier inspection.


In [1]:
from hashlib import sha256
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, mean_absolute_error, mean_squared_error, precision_score, recall_score, r2_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RUN_FINAL_TEST = True  # Change to True only after reviewing frozen_decision.json.
RANDOM_STATE = 42
TARGET_CLASS = "target_high_demand"
TARGET_REGRESSION = "target_next_week_sales"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/data/process_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from fashion-demand-ml.")
sys.path.insert(0, str(ROOT))
output_dir = ROOT / "notebooks" / "closed_test_protocol" / "results"
decision_path = output_dir / "frozen_decision.json"
if not decision_path.is_file():
    raise FileNotFoundError("Run the development notebook and freeze its decision first.")
decision = json.loads(decision_path.read_text(encoding="utf-8"))
processed_dir = ROOT / "data/processed"
pointer = json.loads((processed_dir / "CURRENT.json").read_text(encoding="utf-8"))
run_dir = (processed_dir / pointer["run_dir"]).resolve()
if not run_dir.is_relative_to(processed_dir.resolve()) or pointer.get("preview"):
    raise ValueError("CURRENT.json must select a complete publication.")
dataset_path = run_dir / "article_week.csv.gz"
digest = sha256()
with dataset_path.open("rb") as stream:
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
        digest.update(block)
if decision["publication_run"] != run_dir.name or decision["publication_sha256"] != digest.hexdigest():
    raise ValueError("The publication changed after model selection.")
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
HIGH_DEMAND_THRESHOLD = int(summary["classification_target"]["threshold_units"])
NUMERIC_FEATURES = [
    "sales", "lag_1_sales", "lag_2_sales", "lag_4_sales", "lag_12_sales",
    "rolling_4w_sales_mean", "rolling_4w_sales_std", "rolling_4w_active_weeks",
    "rolling_12w_sales_mean", "rolling_12w_sales_std", "rolling_12w_active_weeks",
    "sales_trend_4w", "mean_price", "rolling_4w_mean_price",
    "price_change_from_4w_mean", "rolling_4w_online_share", "week_sin", "week_cos",
]
CATEGORICAL_FEATURES = [
    "product_type_name", "product_group_name", "index_group_name",
    "section_name", "garment_group_name", "colour_group_name",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


In [2]:
NUMERIC_FEATURES = [
    "sales", "lag_1_sales", "lag_2_sales", "lag_4_sales", "lag_12_sales",
    "rolling_4w_sales_mean", "rolling_4w_sales_std", "rolling_4w_active_weeks",
    "rolling_12w_sales_mean", "rolling_12w_sales_std", "rolling_12w_active_weeks",
    "sales_trend_4w", "mean_price", "rolling_4w_mean_price",
    "price_change_from_4w_mean", "rolling_4w_online_share", "week_sin", "week_cos",
]
FORBIDDEN_PREDICTORS = {
    "article_id", "product_code", TARGET_REGRESSION, TARGET_CLASS,
    "target_week", "split", "target_is_observed", "history_is_observed",
}
if FORBIDDEN_PREDICTORS.intersection(FEATURES):
    raise ValueError("A target, identifier, or future-period field is present among predictors.")
if set(FEATURES) != set(NUMERIC_FEATURES + CATEGORICAL_FEATURES):
    raise ValueError("Feature definitions are inconsistent.")

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=25,
                max_categories=80,
                sparse_output=True,
            ),
        ),
    ]
)
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# Histogram boosting needs a compact, dense table with categorical columns encoded
# as integers. This preparation is learned separately within every training fold.
boosting_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            SimpleImputer(strategy="median", keep_empty_features=True),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(
                        strategy="constant", fill_value="Unknown",
                        keep_empty_features=True,
                    )),
                    ("ordinal", OrdinalEncoder(
                        handle_unknown="use_encoded_value", unknown_value=-1,
                        min_frequency=25, max_categories=80,
                    )),
                ]
            ),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    sparse_threshold=0,
)

def feature_frame(frame):
    missing = sorted(set(FEATURES) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing predictor columns: {missing}")
    return frame.loc[:, FEATURES].copy()

def rows_for_weeks(frame, weeks):
    return frame.loc[frame["target_week"].isin(weeks)]

def make_pipeline(estimator):
    chosen_preprocessor = (
        boosting_preprocessor
        if isinstance(estimator, HistGradientBoostingRegressor)
        else preprocessor
    )
    return Pipeline(
        [("preprocessor", clone(chosen_preprocessor)), ("estimator", clone(estimator))]
    )

def regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    raw_prediction = np.asarray(predicted, dtype=float)
    negative_prediction_share = float(np.mean(raw_prediction < 0))
    prediction = np.maximum(raw_prediction, 0)
    return {
        "MAE": mean_absolute_error(actual, prediction),
        "RMSE": mean_squared_error(actual, prediction) ** 0.5,
        "R2": r2_score(actual, prediction),
        "negative_raw_prediction_share": negative_prediction_share,
    }

def classification_metrics(actual, probability):
    actual = np.asarray(actual, dtype=int)
    probability = np.asarray(probability, dtype=float)
    predicted = probability >= 0.5
    return {
        "precision_at_0.5": precision_score(actual, predicted, zero_division=0),
        "recall_at_0.5": recall_score(actual, predicted, zero_division=0),
        "F1_at_0.5": f1_score(actual, predicted, zero_division=0),
        "PR-AUC": average_precision_score(actual, probability),
        "ROC-AUC": roc_auc_score(actual, probability),
        "Brier": brier_score_loss(actual, probability),
    }

print(f"Numeric predictors: {len(NUMERIC_FEATURES)}")
print(f"Categorical predictors: {len(CATEGORICAL_FEATURES)}")
print("All preprocessing is inside each fold-specific pipeline; boosting uses compact categorical encoding.")


# Threshold choice uses ONLY inner-validation predictions.
THRESHOLD_GRID = np.linspace(0.0, 1.0, 101)

def threshold_metrics(actual, probability, threshold):
    actual = np.asarray(actual, dtype=int)
    probability = np.asarray(probability, dtype=float)
    predicted = probability >= float(threshold)
    return {
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted, zero_division=0),
        "F1": f1_score(actual, predicted, zero_division=0),
    }

def select_inner_threshold(inner_predictions):
    """Maximise mean F1 across inner folds; tie: mean precision, then higher cut."""
    if not inner_predictions:
        raise ValueError("No inner-validation predictions available.")
    rows = []
    for threshold in THRESHOLD_GRID:
        scores = [threshold_metrics(y, p, threshold) for y, p in inner_predictions]
        rows.append({
            "threshold": float(threshold),
            "mean_inner_F1": float(np.mean([s["F1"] for s in scores])),
            "mean_inner_precision": float(np.mean([s["precision"] for s in scores])),
            "mean_inner_recall": float(np.mean([s["recall"] for s in scores])),
        })
    sweep = pd.DataFrame(rows)
    tied = sweep.loc[np.isclose(sweep["mean_inner_F1"], sweep["mean_inner_F1"].max())]
    best = tied.sort_values(
        ["mean_inner_precision", "threshold"], ascending=[False, False]
    ).iloc[0]
    return float(best["threshold"]), sweep


Numeric predictors: 18
Categorical predictors: 6
All preprocessing is inside each fold-specific pipeline; boosting uses compact categorical encoding.


In [3]:
REGRESSION_CANDIDATES = {
    "ridge_alpha_1": Ridge(alpha=1.0, solver="lsqr"),
    "ridge_alpha_10": Ridge(alpha=10.0, solver="lsqr"),
    "tree_depth_8": DecisionTreeRegressor(
        max_depth=8, min_samples_leaf=100, random_state=RANDOM_STATE
    ),
    "tree_depth_12": DecisionTreeRegressor(
        max_depth=12, min_samples_leaf=100, random_state=RANDOM_STATE
    ),
    "random_forest_depth_12": RandomForestRegressor(
        n_estimators=30, max_depth=12, min_samples_leaf=100,
        max_features=0.7, n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "boosting_squared_error": HistGradientBoostingRegressor(
        loss="squared_error", max_iter=100, max_leaf_nodes=31,
        learning_rate=0.08, early_stopping=False,
        categorical_features=list(range(len(NUMERIC_FEATURES), len(FEATURES))),
        random_state=RANDOM_STATE,
    ),
    "boosting_absolute_error": HistGradientBoostingRegressor(
        loss="absolute_error", max_iter=100, max_leaf_nodes=31,
        learning_rate=0.08, early_stopping=False,
        categorical_features=list(range(len(NUMERIC_FEATURES), len(FEATURES))),
        random_state=RANDOM_STATE,
    ),
    "boosting_poisson": HistGradientBoostingRegressor(
        loss="poisson", max_iter=100, max_leaf_nodes=31,
        learning_rate=0.08, early_stopping=False,
        categorical_features=list(range(len(NUMERIC_FEATURES), len(FEATURES))),
        random_state=RANDOM_STATE,
    ),
}


CLASSIFICATION_CANDIDATES = {
    "logistic_C_0.1": LogisticRegression(C=0.1, solver="liblinear", max_iter=250, random_state=RANDOM_STATE),
    "logistic_C_1": LogisticRegression(C=1.0, solver="liblinear", max_iter=250, random_state=RANDOM_STATE),
    "tree_depth_8": DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, class_weight="balanced", random_state=RANDOM_STATE),
    "tree_depth_12": DecisionTreeClassifier(max_depth=12, min_samples_leaf=100, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest_depth_12": RandomForestClassifier(n_estimators=30, max_depth=12, min_samples_leaf=100, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
}



In [4]:
if not RUN_FINAL_TEST:
    raise RuntimeError("The test remains closed. Set RUN_FINAL_TEST=True only when the development decision is final.")
final_path = output_dir / "final_test_metrics.json"
if final_path.exists():
    raise FileExistsError("Final test metrics already exist. This notebook is deliberately one-shot.")
columns = list(dict.fromkeys([
    "split", "target_week", "iso_week", TARGET_REGRESSION, TARGET_CLASS,
    *[c for c in NUMERIC_FEATURES if c not in ("week_sin", "week_cos")],
    *CATEGORICAL_FEATURES,
]))
parts = []
for chunk in pd.read_csv(dataset_path, usecols=columns,
                         parse_dates=["target_week"], chunksize=250_000):
    parts.append(chunk.loc[chunk["split"].isin(["train", "validation", "test"])].copy())
full_data = pd.concat(parts, ignore_index=True)
for partition in ("train", "validation", "test"):
    actual = int(full_data["split"].eq(partition).sum())
    expected = int(summary["split_counts"][partition])
    if actual != expected:
        raise ValueError(f"{partition}: {actual} rows, expected {expected}")
if full_data[["split", "target_week", TARGET_REGRESSION, TARGET_CLASS]].isna().any().any():
    raise ValueError("Missing required target or partition values.")
if not full_data[TARGET_CLASS].eq(full_data[TARGET_REGRESSION].ge(HIGH_DEMAND_THRESHOLD).astype("int8")).all():
    raise ValueError("The published classification target is inconsistent.")
full_data["iso_week"] = full_data["iso_week"].astype(int)
full_data["week_sin"] = np.sin(2 * np.pi * (full_data["iso_week"] - 1) / 52.1775)
full_data["week_cos"] = np.cos(2 * np.pi * (full_data["iso_week"] - 1) / 52.1775)
for column in CATEGORICAL_FEATURES:
    full_data[column] = full_data[column].fillna("Unknown").astype(str)
full_development = full_data.loc[full_data["split"].isin(["train", "validation"])].copy()
full_test = full_data.loc[full_data["split"].eq("test")].copy()
if (full_development[TARGET_REGRESSION] < 0).any():
    raise ValueError("Regression targets must be nonnegative.")
if full_development["target_week"].max() >= full_test["target_week"].min():
    raise ValueError("Development and test periods overlap.")
regression_name = decision["regression_candidate"]
classification_name = decision["classification_candidate"]
threshold = float(decision["probability_threshold"])
if regression_name not in REGRESSION_CANDIDATES or classification_name not in CLASSIFICATION_CANDIDATES:
    raise ValueError("The frozen decision names an unknown model.")
regressor = make_pipeline(REGRESSION_CANDIDATES[regression_name])
classifier = make_pipeline(CLASSIFICATION_CANDIDATES[classification_name])
regressor.fit(feature_frame(full_development), full_development[TARGET_REGRESSION])
classifier.fit(feature_frame(full_development), full_development[TARGET_CLASS].to_numpy(dtype="int8"))
regression_prediction = regressor.predict(feature_frame(full_test))
classification_probability = classifier.predict_proba(feature_frame(full_test))[:, 1]
regression_result = regression_metrics(full_test[TARGET_REGRESSION], regression_prediction)
classification_result = classification_metrics(full_test[TARGET_CLASS], classification_probability)
threshold_result = threshold_metrics(full_test[TARGET_CLASS], classification_probability, threshold)
persistence_result = regression_metrics(full_test[TARGET_REGRESSION], full_test["sales"])
current_sales_probability = full_test["sales"].ge(HIGH_DEMAND_THRESHOLD).astype(float).to_numpy()
current_sales_rule_result = classification_metrics(full_test[TARGET_CLASS], current_sales_probability)
record = {
    "publication_run": run_dir.name,
    "publication_sha256": digest.hexdigest(),
    "development_rows": len(full_development), "test_rows": len(full_test),
    "frozen_decision": decision,
    "regression": regression_result,
    "persistence_baseline": persistence_result,
    "classification_probability_metrics": classification_result,
    "classification_at_frozen_threshold": threshold_result,
    "current_sales_rule": current_sales_rule_result,
}
final_path.write_text(json.dumps(record, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
print("Final test evaluated once. Rows:", len(full_test))
print("Regression:", regression_result)
print("Persistence baseline:", persistence_result)
print("Classification at frozen threshold:", threshold_result)
print("Probability metrics:", {key: classification_result[key] for key in ["PR-AUC", "ROC-AUC", "Brier"]})
print("Current-sales rule:", current_sales_rule_result)
print("Saved:", final_path)


Final test evaluated once. Rows: 289490
Regression: {'MAE': 6.647704681979787, 'RMSE': 21.90737757172193, 'R2': 0.6329712067887893, 'negative_raw_prediction_share': 0.009437286262047049}
Persistence baseline: {'MAE': 7.8489412414936615, 'RMSE': 24.34022381904405, 'R2': 0.5469266850034519, 'negative_raw_prediction_share': 0.0}
Classification at frozen threshold: {'precision': 0.7785790961733267, 'recall': 0.8414241597359521, 'F1': 0.8087826483014245}
Probability metrics: {'PR-AUC': 0.8927317079556382, 'ROC-AUC': 0.9595131464342507, 'Brier': 0.058252694851719924}
Current-sales rule: {'precision_at_0.5': 0.7824256855365236, 'recall_at_0.5': 0.8322086240420908, 'F1_at_0.5': 0.806549692785203, 'PR-AUC': 0.686614131076331, 'ROC-AUC': 0.8850844205676685, 'Brier': 0.08439669764067843}
Saved: c:\Iratxe\UPM\prdl y mllb\fashion-demand-ml\notebooks\closed_test_protocol\results\final_test_metrics.json


In [5]:
import os
print(os.getpid())

21968


In [6]:
exec((ROOT / "notebooks/closed_test_protocol/save_predictions_current_kernel.py").read_text(encoding="utf-8"))

Saved existing final-test predictions: c:\Iratxe\UPM\prdl y mllb\fashion-demand-ml\notebooks\closed_test_protocol\results\final_test_predictions.csv.gz
No model was retrained.
